# 16 · Final manuscript consistency and compilation audit — Gate 17

**Purpose.** Audit the frozen evidence package after Gate 16 without re-estimating any model.

This notebook verifies:

- passed dependencies from Gates 14–16;
- numerical agreement between frozen CSV outputs, manuscript, and supplement;
- abstract length and absence of numerical detail;
- figure, citation, label, and reference completeness;
- clean LaTeX compilation of the manuscript and supplementary material;
- remaining author-dependent submission decisions separately from computational validity.

Gate 17 may pass while the submission-readiness section still records unresolved authorship, funding, repository DOI, or final-title decisions.


In [3]:
%pip -q install pypdf pymupdf


In [4]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, re, shutil, subprocess

import numpy as np
import pandas as pd

ROOT = Path("/content/colombia-stochastic-access")
PAPER = ROOT / "paper"
BUILD = PAPER / "build"
OUT = ROOT / "outputs/bogota_em2023/gate17"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
for directory in (BUILD, OUT):
    directory.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Environment outside Colab or Drive not mounted: {exc}")

if shutil.which("latexmk") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run([
        "apt-get", "install", "-y", "-qq",
        "latexmk", "texlive-latex-base", "texlive-latex-recommended",
        "texlive-latex-extra", "texlive-fonts-recommended",
    ], check=True)

G14 = Path(os.environ.get("GATE14_DIR_OVERRIDE", DRIVE_BASE / "results/gate14"))
G15 = Path(os.environ.get("GATE15_DIR_OVERRIDE", DRIVE_BASE / "results/gate15"))
G16 = Path(os.environ.get("GATE16_DIR_OVERRIDE", DRIVE_BASE / "results/gate16"))
paths = {
    "gate14": G14 / "gate_14.json",
    "gate15": G15 / "gate_15.json",
    "gate16": G16 / "gate_16.json",
    "main_table": G14 / "table_1_main_scope_sex_gaps.csv",
    "amplification_table": G14 / "table_2_paired_scope_amplification.csv",
    "scope_points": G16 / "boundary_multiscale_point_estimates.csv",
    "slope_intervals": G16 / "boundary_multiscale_slope_intervals.csv",
    "contrast_intervals": G16 / "boundary_multiscale_contrast_intervals.csv",
    "manuscript": PAPER / "manuscript.tex",
    "supplement": PAPER / "supplementary_material.tex",
    "bibliography": PAPER / "references.bib",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing frozen inputs: " + ", ".join(missing))

reports = {
    number: json.loads(paths[f"gate{number}"].read_text(encoding="utf-8"))
    for number in (14, 15, 16)
}
for number, report in reports.items():
    if not report.get("gate", {}).get(f"gate_{number}_passed", False):
        raise RuntimeError(f"Gate {number} is not approved")

print("Validated dependencies: Gates 14, 15, and 16")
print("Output:", OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Missing frozen inputs: /content/colombia-stochastic-access/paper/manuscript.tex, /content/colombia-stochastic-access/paper/supplementary_material.tex, /content/colombia-stochastic-access/paper/references.bib

## Numerical and source consistency

All values are read from frozen outputs. The audit searches the rounded values actually reported in the manuscript and supplement; it does not recompute any estimand.


In [ ]:
main_table = pd.read_csv(paths["main_table"])
amplification = pd.read_csv(paths["amplification_table"])
scope_points = pd.read_csv(paths["scope_points"])
slopes = pd.read_csv(paths["slope_intervals"])
contrasts = pd.read_csv(paths["contrast_intervals"])
manuscript = paths["manuscript"].read_text(encoding="utf-8")
supplement = paths["supplement"].read_text(encoding="utf-8")
bibliography = paths["bibliography"].read_text(encoding="utf-8")

abstract_match = re.search(
    r"\\begin\{abstract\}(.*?)\\end\{abstract\}",
    manuscript, flags=re.DOTALL,
)
if abstract_match is None:
    raise RuntimeError("Abstract not found")
abstract_plain = re.sub(r"\\[A-Za-z]+|[{}$]", " ", abstract_match.group(1))
abstract_words = re.findall(r"[A-Za-zÀ-ÿ]+(?:[-'][A-Za-zÀ-ÿ]+)*", abstract_plain)

figure_names = [
    "figure_1_scope_sex_gaps.png",
    "figure_2_paired_scope_amplification.png",
    "figure_3_external_municipality_kernel_contributions.png",
    "figure_boundary_response_total_gap.png",
    "figure_boundary_response_decomposition.png",
]
figure_paths = [PAPER / "figures" / name for name in figure_names]

bib_keys = set(re.findall(r"@\w+\s*\{\s*([^,]+),", bibliography))
cited_keys = set()
for group in re.findall(r"\\cite[pt]?\{([^}]+)\}", manuscript):
    cited_keys.update(key.strip() for key in group.split(","))

labels = set(re.findall(r"\\label\{([^}]+)\}", manuscript + supplement))
refs = set(re.findall(r"\\ref\{([^}]+)\}", manuscript + supplement))

def rounded_tokens(frame, columns):
    tokens = []
    for column in columns:
        tokens.extend(f"{value:.3f}" for value in pd.to_numeric(frame[column]).dropna())
    return sorted(set(tokens))

slope_tokens = rounded_tokens(
    slopes,
    ["point_slope", "simultaneous_ci_lower", "simultaneous_ci_upper"],
)
manuscript_slope_tokens = rounded_tokens(
    slopes[slopes["effect"].isin([
        "female_minus_male_total", "kernel_contribution",
    ])],
    ["point_slope", "simultaneous_ci_lower", "simultaneous_ci_upper"],
)
endpoint_tokens = [
    "0.999", "0.827", "0.705", "0.479", "0.399", "0.240",
    "0.520", "0.429", "0.465",
]
monotonicity_tokens = ["0.756", "0.110", "0.212"]

checks = {
    "gate14_dependency_valid": True,
    "gate15_dependency_valid": True,
    "gate16_dependency_valid": True,
    "all_loaded_tables_finite": bool(all(
        np.isfinite(frame.select_dtypes(include="number").to_numpy()).all()
        for frame in (main_table, amplification, scope_points, slopes, contrasts)
    )),
    "abstract_present": bool(abstract_match),
    "abstract_between_150_and_250_words": bool(150 <= len(abstract_words) <= 250),
    "abstract_contains_no_digits": bool(re.search(r"\d", abstract_match.group(1)) is None),
    "all_five_figures_present": bool(all(path.exists() and path.stat().st_size > 0 for path in figure_paths)),
    "all_citations_resolved": bool(cited_keys <= bib_keys),
    "all_cross_references_resolved": bool(refs <= labels),
    "endpoint_claims_match_frozen_outputs": bool(all(token in manuscript for token in endpoint_tokens)),
    "gate16_slope_claims_match_manuscript": bool(all(token in manuscript for token in manuscript_slope_tokens)),
    "gate16_slope_claims_match_supplement": bool(all(token in supplement for token in slope_tokens)),
    "monotonicity_probabilities_reported": bool(all(token in manuscript + supplement for token in monotonicity_tokens)),
    "causal_boundary_stated": bool(
        "not a causal or continuous boundary effect" in manuscript
        and "does not identify causal effects" in manuscript
    ),
    "strict_monotonicity_not_overclaimed": bool(
        "not strict stepwise monotonicity" in manuscript
        and "not universal local monotonicity" in manuscript
    ),
}
display(pd.Series(checks, name="pre_compilation_result"))
print("Abstract words:", len(abstract_words))
print("Missing citation keys:", sorted(cited_keys - bib_keys))
print("Missing labels:", sorted(refs - labels))


## Reproducible LaTeX compilation

Both documents are compiled from the current repository state. Gate 17 rejects LaTeX errors, undefined citations, and undefined cross-references.


In [ ]:
def compile_tex(filename):
    command = [
        "latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error",
        f"-outdir={BUILD}", filename,
    ]
    completed = subprocess.run(
        command, cwd=PAPER, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    log_path = BUILD / Path(filename).with_suffix(".log").name
    log_text = log_path.read_text(encoding="utf-8", errors="replace") if log_path.exists() else completed.stdout
    return {
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "log": log_text,
        "pdf": BUILD / Path(filename).with_suffix(".pdf").name,
    }

compiled = {
    "manuscript": compile_tex("manuscript.tex"),
    "supplement": compile_tex("supplementary_material.tex"),
}
for name, result in compiled.items():
    if result["returncode"] != 0:
        print(f"--- {name} compilation tail ---")
        print(result["stdout"][-5000:])

from pypdf import PdfReader
for name, result in compiled.items():
    checks[f"{name}_compiles"] = bool(
        result["returncode"] == 0
        and result["pdf"].exists()
        and result["pdf"].stat().st_size > 0
    )
    checks[f"{name}_has_no_latex_errors"] = "LaTeX Error" not in result["log"]
    checks[f"{name}_has_no_undefined_references"] = not bool(re.search(
        r"(undefined citations|undefined references|Citation .* undefined|Reference .* undefined)",
        result["log"], flags=re.IGNORECASE,
    ))

pdf_summary = []
for name, result in compiled.items():
    if result["pdf"].exists():
        pdf_summary.append({
            "document": name,
            "pages": len(PdfReader(str(result["pdf"])).pages),
            "bytes": result["pdf"].stat().st_size,
        })
display(pd.DataFrame(pdf_summary))


## Gate 17 report and visual preview

Computational validity is separated from author-dependent submission readiness. The latter remains a checklist rather than a reason to invalidate the analysis.


In [ ]:
checks["gate_17_passed"] = bool(all(checks.values()))

submission_readiness = {
    "final_title_selected": False,
    "funding_statement_confirmed": "[Confirm before submission.]" not in manuscript,
    "authorship_finalized": "[Update if co-authors are added.]" not in manuscript,
    "acknowledgements_finalized": "[Update before submission.]" not in manuscript,
    "repository_doi_inserted": "persistent identifier inserted here" not in manuscript,
    "ai_disclosure_finalized": "should be revised" not in manuscript,
}
submission_readiness["submission_ready"] = bool(
    checks["gate_17_passed"] and all(submission_readiness.values())
)

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "final frozen-evidence, manuscript, supplement, figure, and compilation audit",
    "dependencies": [14, 15, 16],
    "abstract_words": len(abstract_words),
    "compiled_documents": pdf_summary,
    "gate": checks,
    "submission_readiness": submission_readiness,
}
(OUT / "gate_17.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
pd.Series(checks, name="result").to_csv(OUT / "gate_17_checks.csv")
pd.Series(submission_readiness, name="result").to_csv(
    OUT / "submission_readiness_checks.csv"
)

persistent = DRIVE_BASE / "results/gate17"
persistent.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent / artifact.name)
for result in compiled.values():
    if result["pdf"].exists():
        shutil.copy2(result["pdf"], persistent / result["pdf"].name)

display(pd.Series(checks, name="result"))
display(pd.Series(submission_readiness, name="result"))
print("Gate 17:", "PASSED" if checks["gate_17_passed"] else "FAILED")
print("Submission ready:", submission_readiness["submission_ready"])
print("Persistent copy:", persistent)


In [ ]:
import fitz
import matplotlib.pyplot as plt

for name, result in compiled.items():
    if not result["pdf"].exists():
        continue
    document = fitz.open(result["pdf"])
    images = []
    for page in document:
        pix = page.get_pixmap(matrix=fitz.Matrix(0.45, 0.45), alpha=False)
        images.append(np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n))
    columns = 3
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(10, 4.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for index, axis in enumerate(axes):
        axis.axis("off")
        if index < len(images):
            axis.imshow(images[index])
            axis.set_title(f"{name} — page {index + 1}")
    fig.tight_layout()
    preview = OUT / f"{name}_contact_sheet.png"
    fig.savefig(preview, dpi=180, bbox_inches="tight")
    shutil.copy2(preview, persistent / preview.name)
    plt.show()
